<a href="https://colab.research.google.com/github/arman-hossain45/ML_Pipe_Line/blob/main/datathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Classification algorithms
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier

import pickle
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings("ignore")

# ১. ডেটা লোড করা (Loading datasets)
train_df = pd.read_csv("dataset.csv")
test_df = pd.read_csv("test.csv")

# অপ্রয়োজনীয় ID কলাম বাদ দেওয়া
train_df = train_df.drop(columns=['Order', 'PID'], errors='ignore')
test_df_ids = test_df[['Order', 'PID']].copy() if 'Order' in test_df.columns else None
test_df = test_df.drop(columns=['Order', 'PID'], errors='ignore')

# ২. টার্গেট ভ্যারিয়েবলকে ক্লাসিফিকেশনে রূপান্তর করা
# যেহেতু মূল ডেটাতে SalePrice সংখ্যাবাচক, ক্লাসিফিকেশনের জন্য এটিকে Median দিয়ে High/Low করা হলো
median_price = train_df['SalePrice'].median()
train_df['Price_Class'] = np.where(train_df['SalePrice'] >= median_price, 'High', 'Low')
train_df = train_df.drop(columns=['SalePrice'])

# ৩. সংখ্যাবাচক ও ক্যাটাগরিকাল ফিচার আলাদা করা
x = train_df.drop('Price_Class', axis=1)
y = train_df['Price_Class']

numerical_feature = x.select_dtypes(include=['int64', 'float64'])
categorical_feature = x.select_dtypes(include=['object'])

# ৪. আউটলায়ার হ্যান্ডেল করা (IQR Capping পদ্ধতি)
x_cleaned = x.copy()
for col in numerical_feature.columns:
    q1 = x_cleaned[col].quantile(0.25)
    q3 = x_cleaned[col].quantile(0.75)
    IQR = q3 - q1
    lower = q1 - 1.5 * IQR
    upper = q3 + 1.5 * IQR
    x_cleaned[col] = x_cleaned[col].clip(lower, upper)

# ৫. টার্গেট কলাম এনকোড করা
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# ৬. পাইপলাইন এবং কলাম ট্রান্সফরমার তৈরি
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # আউটলায়ারের জন্য median উত্তম
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numerical_feature.columns),
    ('cat', cat_transformer, categorical_feature.columns)
])

# ৭. ট্রেন-টেস্ট স্প্লিট
x_train, x_test, y_train, y_test = train_test_split(x_cleaned, y_encoded, test_size=0.2, random_state=42)

# ৮. বেস লার্নার মডেল ডিফাইন করা
lr = LogisticRegression(max_iter=2000, class_weight='balanced')
dt = DecisionTreeClassifier(max_depth=5, min_samples_split=5, random_state=42)
rf = RandomForestClassifier(max_depth=10, min_samples_leaf=5, random_state=42)
knn = KNeighborsClassifier(n_neighbors=15, p=2, metric='minkowski')
svm = SVC(kernel='rbf', gamma='scale', C=5, random_state=42) # রৈখিক না হলে rbf ভালো কাজ করে
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, subsample=0.8, random_state=42)

# Voting and Stacking Classifier
voting_class = VotingClassifier(
    estimators=[("lr", lr), ('dt', dt), ('rf', rf), ('knn', knn), ('svm', svm), ('gb', gb)],
    voting='hard'
)

stacking_class = StackingClassifier(
    estimators=[("lr", lr), ('dt', dt), ('rf', rf), ('knn', knn), ('svm', svm), ('gb', gb)],
    final_estimator=RidgeClassifier()
)

model_to_train = {
    'Logistic Regression': lr,
    'Decision Tree': dt,
    'Random Forest': rf,
    'KNN': knn,
    'SVM': svm,
    'Gradient Boosting': gb,
    "Voting Classifier": voting_class,
    "Stacking Class": stacking_class
}

# ৯. মডেল ট্রেনিং এবং মূল্যায়ন লুপ
result = []
for model_name, model in model_to_train.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(x_train, y_train)
    y_pred = pipe.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)

    result.append({"Model": model_name, "accuracy": accuracy})

res = pd.DataFrame(result).sort_values(by='accuracy', ascending=False)
print("--- মডেল ইভালুয়েশন রেজাল্ট ---")
print(res)

# ১০. সেরা মডেল নিয়ে ফাইনাল পাইপলাইন তৈরি ও ক্রস-ভ্যালিডেশন (উদাহরণস্বরূপ Random Forest)
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(max_depth=None, max_features='sqrt', n_estimators=50, random_state=42))
])

cv_result = cross_val_score(rf_pipeline, x_train, y_train, cv=5, scoring='accuracy')
print(f"\n৫-ফোল্ড ক্রস ভ্যালিডেশন স্কোর: {cv_result}")
print(f"গড় অ্যাকুরেসি: {cv_result.mean():.4f}")

# ১১. গ্রিড সার্চ সিভি (Hyperparameter Tuning)
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [None, 10],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [2, 4]
}

grid_search_rf = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=3,  # দ্রুত রান করার জন্য ৩ দেওয়া হয়েছে
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search_rf.fit(x_train, y_train)

print(f"\nসেরা প্যারামিটারসমূহ: {grid_search_rf.best_params_}")
print(f"সেরা স্কোর: {grid_search_rf.best_score_:.4f}")

# ১২. বেস্ট মডেল দিয়ে টেস্ট ডেটা প্রেডিকশন এবং পারফরম্যান্স রিপোর্ট
best_rf_model = grid_search_rf.best_estimator_
y_final_pred = best_rf_model.predict(x_test)

print("\n--- কনফিউশন ম্যাট্রিক্স ---")
print(confusion_matrix(y_test, y_final_pred))
print("\n--- ক্লাসিফিকেশন রিপোর্ট ---")
print(classification_report(y_test, y_final_pred))

# ১৩. মডেল সেভ করা এবং লোড করে টেস্ট ফাইলের (.csv) ওপর প্রেডিক্ট করা
filename = 'house_classification_model.pkl'
with open(filename, 'wb') as file:
    pickle.dump(best_rf_model, file)

# সেভ করা মডেল পুনরায় লোড করা
with open(filename, 'rb') as file:
    loaded_model = pickle.load(file)

# আনসিন 'test.csv' ডেটার ওপর ফাইনাল প্রেডিকশন
test_preds = loaded_model.predict(test_df)
print("\ntest.csv ফাইলের প্রথম ১০টি প্রেডিকশন (0=Low, 1=High):")
print(test_preds[:10])

--- মডেল ইভালুয়েশন রেজাল্ট ---
                 Model  accuracy
6    Voting Classifier  0.951705
4                  SVM  0.943182
2        Random Forest  0.940341
0  Logistic Regression  0.937500
7       Stacking Class  0.937500
5    Gradient Boosting  0.934659
3                  KNN  0.931818
1        Decision Tree  0.869318

৫-ফোল্ড ক্রস ভ্যালিডেশন স্কোর: [0.91134752 0.93594306 0.92882562 0.88256228 0.88967972]
গড় অ্যাকুরেসি: 0.9097
Fitting 3 folds for each of 16 candidates, totalling 48 fits

সেরা প্যারামিটারসমূহ: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 50}
সেরা স্কোর: 0.9168

--- কনফিউশন ম্যাট্রিক্স ---
[[165  13]
 [  4 170]]

--- ক্লাসিফিকেশন রিপোর্ট ---
              precision    recall  f1-score   support

           0       0.98      0.93      0.95       178
           1       0.93      0.98      0.95       174

    accuracy                           0.95       352
   macro avg       0.95      0.95      0.95

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train_df = pd.read_csv('dataset.csv')

In [ ]:
test_df

In [ ]:
train_df.head(10)

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1323,902405070,70,RM,60.0,10800,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,184000
1,2476,531376030,60,RL,65.0,7800,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2006,WD,Normal,184000
2,2091,906201022,20,RL,114.0,10357,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,179900
3,2734,905403050,50,RL,62.0,6488,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,3,2006,WD,Family,128000
4,2882,911175440,190,C (all),50.0,9000,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,10,2006,WD,Abnorml,115000
5,1557,911102180,50,C (all),52.0,5150,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,5,2008,WD,Normal,80900
6,2529,534129230,60,RL,80.0,10400,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,11,2006,WD,Normal,165150
7,673,535403280,20,RL,64.0,8712,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2009,WD,Normal,153000
8,1912,535102010,85,RL,NaN,10050,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,6,2007,WD,Normal,175000
9,2385,528114010,20,RL,120.0,14780,Pave,NaN,IR1,HLS,...,0,NaN,NaN,NaN,0,6,2006,WD,Normal,415000
